In [ ]:
import xarray as xr

ds = xr.open_dataset("data/davis-TUD-EWI_Roof_Electrical_Engineering_202501.nc")
print(ds)                  # dims/coords/vars overview
print(ds.variables.keys()) # variable names
print(ds['time'])          # time coordinate

In [6]:
import xarray as xr

paths = [
    "data/davis-TUD-EWI_Roof_Electrical_Engineering_202501.nc",
    "data/davis-TUD-EWI_Roof_Electrical_Engineering_202502.nc",
]
dss = [xr.open_dataset(p) for p in paths]             # eager load
ds  = xr.concat(dss, dim="time").sortby("time")
print(ds)

<xarray.Dataset> Size: 12MB
Dimensions:                   (time: 83522)
Dimensions without coordinates: time
Data variables: (12/16)
    epoch_time                (time) int32 334kB 1735689600 ... 1740787200
    time_as_string            (time) <U19 6MB '2025-01-01' ... '2025-03-01'
    latitude                  (time) float32 334kB 52.0 52.0 52.0 ... 52.0 52.0
    longitude                 (time) float32 334kB 4.373 4.373 ... 4.373 4.373
    altitude                  (time) float32 334kB 100.0 100.0 ... 100.0 100.0
    pressure                  (time) float32 334kB 1.015e+03 ... 1.03e+03
    ...                        ...
    rain_rate                 (time) float32 334kB 0.0 0.0 0.0 ... 0.0 0.0 0.0
    UV_radiation              (time) float32 334kB nan nan nan ... nan nan nan
    wind_speed                (time) float32 334kB 10.73 13.86 ... 1.788 0.447
    wind_to_direction         (time) float32 334kB 337.5 337.5 ... 337.5 337.5
    wind_gust_speed           (time) float32 334kB 17

C:\Users\raf\AppData\Local\Temp\ipykernel_28832\3819922359.py:8: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds  = xr.concat(dss, dim="time").sortby("time")


In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

# ---- 1) open (pick A or B above) ----
# Example: Option B (no dask)
paths = [
    "data/davis-TUD-EWI_Roof_Electrical_Engineering_202501.nc",
    "data/davis-TUD-EWI_Roof_Electrical_Engineering_202502.nc",
]
dss = [xr.open_dataset(p) for p in paths]
ds  = xr.concat(dss, dim="time").sortby("time")

# ---- 2) make timezone-aware timestamps (dataset says time_as_string is UTC) ----
# If ds["time"] is seconds since epoch: prefer that over strings.
if "epoch_time" in ds and "time" in ds:
    t_utc = pd.to_datetime(ds["epoch_time"].values, unit="s", utc=True)
else:
    # fallback: parse the string column
    t_utc = pd.to_datetime(ds["time_as_string"].values).tz_localize("UTC")
t_local = t_utc.tz_convert("Europe/Amsterdam")
ds = ds.assign_coords(time=t_local)

# ---- 3) pick variables and derive wind components ----
# Keep only variables we need (add/remove as you like)
vars_keep = [
    "temperature",
    "wind_speed", "wind_gust_speed",
    "rain", "rain_rate",
]
vars_keep = [v for v in vars_keep if v in ds.variables]
ds_sel = ds[vars_keep]

# wind components (direction is "to": 0°=towards North, 90°=towards East)
if {"wind_speed","wind_to_direction"}.issubset(ds_sel.variables):
    spd = ds_sel["wind_speed"].to_numpy()
    deg = ds_sel["wind_to_direction"].to_numpy()
    th  = np.deg2rad(deg)
    u   = spd * np.sin(th)   # +east
    v   = spd * np.cos(th)   # +north
    ds_sel["wind_u"] = (("time",), u.astype(np.float32))
    ds_sel["wind_v"] = (("time",), v.astype(np.float32))

# ---- 4) to pandas and resample to your bin ----
dfw = ds_sel.to_dataframe()

# numeric columns only
num_cols = dfw.select_dtypes(include=[np.number]).columns

# choose how to aggregate per bin:
# - instantaneous/averaged sensors -> mean (temp, humidity, pressure, wind, irradiance, UV, rain_rate)
# - precip per-interval ("rain" already in mm per 1-minute) -> sum over the bin
how = {c: "mean" for c in num_cols}
if "rain" in num_cols:
    how["rain"] = "sum"

gran = "15min"  # set this to cfg.time_granularity
weather_df = (dfw[num_cols]
              .resample(gran)
              .agg(how)
              .sort_index()
              .interpolate(limit=2)   # small gaps
              .ffill()
              .bfill())

print(weather_df.head())
print(weather_df.columns.tolist())


C:\Users\raf\AppData\Local\Temp\ipykernel_28832\3780715347.py:12: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds  = xr.concat(dss, dim="time").sortby("time")


ValueError: unconverted data remains when parsing with format "%Y-%m-%d": " 00:01:00", at position 1. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.